In [2]:

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

# Load in the data
data_url = '../spambase_data/spambase.data'
names_url = '../spambase_data/spambase.names'

# parse the feature names from the .names file
feature_names = []
with open(names_url, "r") as f:
    for line in f:
        line = line.strip()
        if ':' in line and not line.startswith("|"):
            feature_names.append(line.split(":")[0].strip())
feature_names.append("label") # last column is the class label

df = pd.read_csv(data_url, header=None, names=feature_names)
x = df.drop("label", axis=1).values
y = df["label"].values

# Part 1: K-Fold crosss 
def kfold_cross_validation(x, y, model, k):
    """
    Manual k-fold cross validation.

    steps:
        1. Shuffle and divide data into k equal partitions
        2. For each fold i: train on remaining k-1 folds, validate on fold i
        3. Record validation error on each fold
        4. Return average validation error across all k folds
    """
    n = len(y)
    fold_size = n // k

    # Shuffle indices to ensure random partitioning
    indices = np.random.permutation(n)
    errors = []

    print(f"    {'Fold':<6} {'Val Error':>10} {'Val Acc':>10}")
    print(f"    {'-'*28}")

    for i in range(k):
        # Step 1: define validation and training indices
        val_idx = indices[i * fold_size : (i + 1) * fold_size]
        train_idx = np.concatenate([
            indices[:i * fold_size],
            indices[(i + 1) * fold_size:]
        ])

        # Step 2: split into train and validation sets
        x_train, x_val = x[train_idx], x[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Scale using only training fold statistics (avoid leakage)
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train)
        x_val = scaler.transform(x_val)

        # Step 3: train and evaluate
        model.fit(x_train, y_train)
        y_pred = model.predict(x_val)

        # Step 4: record validation error
        acc = np.mean(y_pred == y_val) 
        err = 1 - acc
        errors.append(err)

        print(f"    {i+1:<6} {err:>} {acc:>10.4f}")

    avg_error = np.mean(errors)
    return avg_error

# Part 2: Run cv for LR and LDA wuth k=5 and k=10
np.random.seed(42)

lr = LogisticRegression(max_iter=2000, solver="lbfgs", C=1.0, random_state=42)
lda = LinearDiscriminantAnalysis()

k_values = [5, 10]
results = {}

for k in k_values:
    print("="*55)
    print(f"    {k}-Fold Cross Validation")
    print("="*55)

    for name, model in [("Logistic Regression", lr), ("LDA", lda)]:
        print(f"\n {name}:")
        avg_err = kfold_cross_validation(x, y, model, k)
        results[(name, k)] = avg_err
        print(f"\n  Average Validition Error: {avg_err:.4f}")
        print(f"    Average Validation Acc : {1-avg_err:.4f}")

# Part 3: Comparison
print("\n" + "=" * 55)
print("  Part 3 - Summary Comparison")
print("=" * 55)
print(f"\n  {'Model':<22} {'k=5 Error':>10} {'k=10 Error':>11}")
print(f"  {'-'*45}")
for name in ["Logistic Regression", "LDA"]:
    e5  = results[(name, 5)]
    e10 = results[(name, 10)]
    print(f"  {name:<22} {e5:>10.4f} {e10:>11.4f}")




    5-Fold Cross Validation

 Logistic Regression:
    Fold    Val Error    Val Acc
    ----------------------------
    1      0.08152173913043481     0.9185
    2      0.07608695652173914     0.9239
    3      0.07934782608695656     0.9207
    4      0.07173913043478264     0.9283
    5      0.07065217391304346     0.9293

  Average Validition Error: 0.0759
    Average Validation Acc : 0.9241

 LDA:
    Fold    Val Error    Val Acc
    ----------------------------
    1      0.10108695652173916     0.8989
    2      0.1119565217391304     0.8880
    3      0.11956521739130432     0.8804
    4      0.1206521739130435     0.8793
    5      0.11086956521739133     0.8891

  Average Validition Error: 0.1128
    Average Validation Acc : 0.8872
    10-Fold Cross Validation

 Logistic Regression:
    Fold    Val Error    Val Acc
    ----------------------------
    1      0.060869565217391286     0.9391
    2      0.05869565217391304     0.9413
    3      0.09999999999999998     0.9000
   

# Observations:
- Logistic Regression outperforms LDA at both k=5 and k=10, confirming the results from Problem 3.
- LDA assumes features are normally distributed with equal covariance across classes. The SPAMBASE features are word/char frequencies that violate this assumption, explaining LDA's weaker performance.
- k=10 generally gives a slightly lower (better) error estimate than k=5 because each validation fold is smaller and the model trains on more data per fold, reducing bias in the error estimate.
- The difference between k=5 and k=10 results is small for both models, suggesting the dataset is large enough that both give stable estimates.